In [1]:
import os

import numpy as np
import pandas as pd

## Leaf area

In [2]:
DIRECTORY = './data/destructive/'
file_list = os.listdir(DIRECTORY)
dataset_list = [file for file in file_list if file.endswith('.xlsx')]
dataset_list.sort()

In [3]:
dataset_list

['destructive(0707)_.xlsx',
 'destructive(0708).xlsx',
 'destructive(0709).xlsx',
 'destructive_CT_(0324-0421-0519-0617).xlsx']

In [4]:
dates = ['0324', '0421', '0522', '0617', '0707', '0708', '0709']

In [5]:
leaf_area_df = {}
_ = 0
for FILENAME in dataset_list[-1:]:
    temp_df = pd.read_excel(DIRECTORY + FILENAME, sheet_name=None, skiprows=0)
    for key in temp_df.keys():
        if temp_df[key].shape[0] == 0:
            continue
            
        leaf_area_df[key] = [temp_df[key]['leaf area'][0]]
    _ += 1 
leaf_area_df = pd.DataFrame.from_dict(leaf_area_df).T
leaf_area_df.columns = ['leaf area']

In [6]:
leaf_areas = []
for _ in range(4):
    leaf_areas.append(leaf_area_df.filter(like=dates[_], axis=0).values.T.squeeze())

In [7]:
leaf_area_df = {}
_ = 0
for FILENAME in dataset_list[:-1]:
    temp_df = pd.read_excel(DIRECTORY + FILENAME, sheet_name=None, skiprows=0)
    for key in temp_df.keys():
        if temp_df[key].shape[0] == 0:
            continue
            
        leaf_area_df[key + '_' + dates[-3:][_]] = [temp_df[key]['leaf area'][0]]
    _ += 1 
leaf_area_df = pd.DataFrame.from_dict(leaf_area_df).T
leaf_area_df.columns = ['leaf area']

In [8]:
a2_df = leaf_area_df.filter(like='A2', axis=0)

In [9]:
for _ in range(3):
    leaf_areas.append(a2_df.filter(like=dates[-3:][_], axis=0).values.T.squeeze())

In [10]:
leaf_areas

[array([2325.75, 2092.14, 1985.18]),
 array([6509.44, 7532.32, 7921.03]),
 array([13240.01, 12620.14, 12134.31]),
 array([15868.04, 14475.55, 14939.53, 15342.76]),
 array([12486.48, 13846.95, 10403.26, 13759.14]),
 array([12861.22, 14610.9 , 10982.65, 12148.78]),
 array([12561.03, 12434.55, 12578.86, 11510.6 , 11054.96, 11971.82,
        11572.67, 11239.36, 13373.94, 11673.69, 11836.51, 12920.71])]

In [11]:
leaf_area_df = pd.DataFrame(leaf_areas, index=['2020-03-24', '2020-04-21', '2020-05-22', '2020-06-17', '2020-07-07', '2020-07-08', '2020-07-09'])

In [12]:
leaf_area_df.index = pd.DatetimeIndex(leaf_area_df.index)

In [13]:
leaf_area_df.to_csv('./results/leaf_area.csv')

## Plant heights & num nodes

In [14]:
DIRECTORY = './data/destructive/'
file_list = os.listdir(DIRECTORY)
dataset_list = [file for file in file_list if file.endswith('.xlsx')]
dataset_list.sort()

In [15]:
dates = ['0324', '0421', '0522', '0617', '0707', '0708', '0709']

In [16]:
plant_height_df = {}
num_nodes_df = {}
for FILENAME in dataset_list[-1:]:
    temp_df = pd.read_excel(DIRECTORY + FILENAME, sheet_name=None, skiprows=1)
    for key in temp_df.keys():
        if temp_df[key].shape[0] == 0:
            continue
        temp_target = temp_df[key].loc[:, 'stem length(cm)':'stem diameter(mm).1'].dropna(axis=0, how='all')
        temp_target.loc[:, 'stem length(cm).1'] = temp_target['stem length(cm).1'].combine_first(temp_target['stem length(cm)'])
        temp_target.loc[:, 'stem diameter(mm).1'] = temp_target['stem diameter(mm).1'].combine_first(temp_target['stem diameter(mm)'])
        
        plant_height_df[key] = max(temp_target.iloc[:, :2].sum())
        num_nodes_df[key] = temp_target.shape[0]
plant_height_df = pd.DataFrame.from_dict([plant_height_df]).T
plant_height_df.columns = ['plant_height']
num_nodes_df = pd.DataFrame.from_dict([num_nodes_df]).T
num_nodes_df.columns = ['num_nodes_df']

In [17]:
plant_height = []
for _ in range(4):
    plant_height.append(plant_height_df.filter(like=dates[_], axis=0).values.T.squeeze())
num_nodes = []
for _ in range(4):
    num_nodes.append(num_nodes_df.filter(like=dates[_], axis=0).values.T.squeeze())

In [18]:
plant_height_df = {}
num_nodes_df = {}
_ = 0
for FILENAME in dataset_list[:-1]:
    temp_df = pd.read_excel(DIRECTORY + FILENAME, sheet_name=None, skiprows=1)
    for key in temp_df.keys():
        if temp_df[key].shape[0] == 0:
            continue
        temp_target = temp_df[key].loc[:, 'internode length':'Unnamed: 12'].dropna(axis=0, how='all')
        try:
            temp_target.loc[:, 'internode length.1'] = temp_target['internode length.1'].combine_first(temp_target['internode length'])
        except KeyError:
            temp_target.loc[:, 'Unnamed: 10'] = temp_target['Unnamed: 10'].combine_first(temp_target['internode length'])
        temp_target.loc[:, 'Unnamed: 12'] = temp_target['Unnamed: 12'].combine_first(temp_target['internode diameter'])
        
        plant_height_df[key + '_' + dates[-3:][_]] = max(temp_target.iloc[:, :2].sum())
        num_nodes_df[key + '_' + dates[-3:][_]] = temp_target.shape[0]
    _ += 1
plant_height_df = pd.DataFrame.from_dict([plant_height_df]).T
plant_height_df.columns = ['plant_height']
num_nodes_df = pd.DataFrame.from_dict([num_nodes_df]).T
num_nodes_df.columns = ['num_nodes_df']

In [19]:
a2_df = plant_height_df.filter(like='A2', axis=0)
a2_df_ = num_nodes_df.filter(like='A2', axis=0)

In [20]:
for _ in range(3):
    plant_height.append(a2_df.filter(like=dates[-3:][_], axis=0).values.T.squeeze())
    num_nodes.append(a2_df_.filter(like=dates[-3:][_], axis=0).values.T.squeeze())

In [21]:
plant_height

[array([51.1, 60.6, 45.7]),
 array([141.4, 129.6, 204.2]),
 array([187.8, 158.3, 198.4]),
 array([201.2, 191.8, 210.1, 217.9]),
 array([202. , 200.1, 200.3, 212.8]),
 array([210.1, 210.2, 193.8, 199.8]),
 array([194.5, 192.3, 196.4, 198.2, 192.7, 209.4, 205.6, 187. , 210. ,
        203.5, 180.9, 194.6])]

In [22]:
num_nodes

[array([14, 15, 14]),
 array([21, 23, 21]),
 array([29, 27, 30]),
 array([31, 33, 31, 30]),
 array([31, 31, 27, 32]),
 array([28, 30, 31, 32]),
 array([29, 30, 31, 31, 32, 31, 34, 32, 35, 32, 40, 34])]

In [23]:
plant_height_df = pd.DataFrame(plant_height, index=['2020-03-24', '2020-04-21', '2020-05-22', '2020-06-17', '2020-07-07', '2020-07-08', '2020-07-09'])
num_nodes_df = pd.DataFrame(num_nodes, index=['2020-03-24', '2020-04-21', '2020-05-22', '2020-06-17', '2020-07-07', '2020-07-08', '2020-07-09'])

In [24]:
plant_height_df.index = pd.DatetimeIndex(plant_height_df.index)
num_nodes_df.index = pd.DatetimeIndex(num_nodes_df.index)

In [25]:
plant_height_df.to_csv('./results/plant_height.csv')
num_nodes_df.to_csv('./results/num_nodes.csv')

## Weights

### Fresh weights

In [26]:
DIRECTORY = './data/destructive/'
file_list = os.listdir(DIRECTORY)
dataset_list = [file for file in file_list if file.endswith('.xlsx')]
dataset_list.sort()

In [27]:
dataset_list

['destructive(0707)_.xlsx',
 'destructive(0708).xlsx',
 'destructive(0709).xlsx',
 'destructive_CT_(0324-0421-0519-0617).xlsx']

In [28]:
FW_df = {}
_ = 0
for FILENAME in dataset_list[-1:]:
    temp_df = pd.read_excel(DIRECTORY + FILENAME, sheet_name=None, skiprows=0)
    for key in temp_df.keys():
        if temp_df[key].shape[0] == 0:
            continue
            
        FW_df[key] = temp_df[key][['Stem FW', 'Leaf FW', 'petiole FW', 'Fruit count', 'Idv fruit FW', 'Plant height']].iloc[:10, :].sum()
    _ += 1 
FW_df = pd.DataFrame.from_dict(FW_df).T

In [29]:
FW_df.index = ['2020-03-24']*3 + ['2020-04-21']*3 + ['2020-05-22']*3 + ['2020-06-17']*4
__ = FW_df

In [30]:
FW_df = {}
_ = 0
for FILENAME in dataset_list[:-1]:
    temp_df = pd.read_excel(DIRECTORY + FILENAME, sheet_name=None, skiprows=0)
    for key in temp_df.keys():
        if temp_df[key].shape[0] == 0:
            continue
            
        FW_df[key] = temp_df[key][['Stem FW', 'Leaf FW', 'petiole FW', 'Fruit count', 'Idv fruit FW', 'Plant height']].iloc[:10, :].sum()
    _ += 1 
FW_df = pd.DataFrame.from_dict(FW_df).T

In [31]:
FW_df = FW_df.filter(like='A2', axis=0)
FW_df.index = ['2020-07-07' for i in range(len(FW_df.index))]

In [32]:
FW_df = pd.concat([__, FW_df], axis=0, sort=False)

### Dry weights

In [33]:
DW_df = {}
_ = 0
for FILENAME in dataset_list[-1:]:
    temp_df = pd.read_excel(DIRECTORY + FILENAME, sheet_name=None, skiprows=22)
    for key in temp_df.keys():
        if temp_df[key].shape[0] == 0:
            continue
            
        DW_df[key] = temp_df[key][['Stem DW', 'Leaf DW', 'petiole DW', 'Fruit DW']].iloc[:3, :].sum()
    _ += 1 
DW_df = pd.DataFrame.from_dict(DW_df).T

In [34]:
DW_df.index = ['2020-03-24']*3 + ['2020-04-21']*3 + ['2020-05-22']*3 + ['2020-06-17']*4
__ = DW_df

In [35]:
DW_df = {}
_ = 0
for FILENAME in dataset_list[:-1]:
    temp_df = pd.read_excel(DIRECTORY + FILENAME, sheet_name=None, skiprows=22)
    for key in temp_df.keys():
        if temp_df[key].shape[0] == 0:
            continue
            
        DW_df[key] = temp_df[key][['Stem DW', 'Leaf DW', 'petiole DW', 'Fruit DW']].iloc[:3, :].sum()
    _ += 1 
DW_df = pd.DataFrame.from_dict(DW_df).T

In [36]:
DW_df = DW_df.filter(like='A2', axis=0)
DW_df.index = ['2020-07-07' for i in range(len(DW_df.index))]

In [37]:
DW_df = pd.concat([__, DW_df], axis=0, sort=False)

In [38]:
weight_df = pd.concat([FW_df, DW_df], axis=1, sort=False)

In [39]:
weight_df.to_csv('./results/weight.csv')